# Electrostatic Capacitance Extraction with Palace

We use `gsim.palace.ElectrostaticSim` to extract the **plate-to-plate (mutual)
capacitance** of the IHP `cmim` (MIM capacitor) cell and compare it against the IHP
PDK compact model. The bottom plate is on Metal5
(terminal `T1` = SPICE `MINUS`), the top plate is the MIM top-metal (`mim` layer,
`T2` = `PLUS`). The two plates are separated by the 40 nm high-k MIM dielectric
(`mim_diel`, SiN); the MIM top-metal is 150 nm and is
brought out through a 10x10 array of Vmim vias to TopMetal1 for external connection.

Palace writes two matrices: `terminal-C.csv` (Maxwell — self-cap on the diagonal,
negative induced charge off-diagonal) and `terminal-Cm.csv` (mutual/"SPICE" — its
off-diagonal `Cm[i,j] = -C[i,j]` is the physical two-terminal capacitor). The MIM cap
value is `Cm[1,2]`, which is exactly what the PDK `cap_cmim` model describes.

The IHP `cap_cmim` is a 2-terminal model with an area + perimeter law (plate-to-plate
only; substrate parasitics are left to extraction). See
[`capacitors_mod.lib`](https://github.com/IHP-GmbH/IHP-Open-PDK/blob/main/ihp-sg13g2/libs.tech/ngspice/models/capacitors_mod.lib).

This notebook:
1. shows that **all** IHP MIM capacitor models report the same 1.5 fF/µm² area
   density (a consistency check — circular, not independent),
2. back-derives an **effective permittivity** for the 40 nm dielectric and
   validates the extraction method on the PDK gate oxide, then
3. sweeps the plate size with that permittivity to show that the residual
   Palace-vs-PDK gap scales like a **fringing field** (smaller for large plates).

**Limitation:** one terminal per layer — multiple electrodes on the same layer (e.g.
interdigitated caps) are not yet supported.

**Requirements:**
- An IHP PDK whose MIM stack carries the split `mim_diel` (dielectric) + `mim`
  (top-metal) layer levels. The pre-fix release lumped both into a single SiO₂
  `mim` level, which cannot model the capacitor (see gdsfactory/IHP#188 and the fix
  in gdsfactory/IHP#225). Install a version containing the fix, e.g.
  `uv pip install "ihp-gdsfactory @ git+https://github.com/gdsfactory/IHP.git"`.
- A Palace backend — either a local `palace` executable (`sim.run_local()`) or a
  [GDSFactory+](https://gdsfactory.com) account for cloud simulation (`sim.run()`).

### Load IHP MIM capacitor

In [ ]:
from ihp import PDK, cells

PDK.activate()

cap_width = 10.0  # um
cap_length = 10.0  # um

# IHP cmim: Metal5 (bottom plate, MINUS) -> 40 nm MIM dielectric (mim_diel, SiN)
#   -> 150 nm MIM top-metal (mim layer, the top electrode) -> 10x10 Vmim vias
#   -> TopMetal1 (external connection, PLUS)
c = cells.cmim(width=cap_width, length=cap_length).copy()
print("Ports:", [(p.name, tuple(p.center)) for p in c.ports])

cc = c.copy()
cc.draw_ports()
cc

### Configure ElectrostaticSim

In [ ]:
from gsim.palace import ElectrostaticSim

sim = ElectrostaticSim()

sim.set_output_dir("./palace-sim-electrostatic")
sim.set_geometry(c)
sim.set_stack(substrate_thickness=2.0)
sim.set_airbox(margin_x=5, margin_y=5, z_above=5, z_below=5)

# Metal5 = bottom plate (MINUS); the `mim` layer is the MIM top plate (PLUS).
# We measure the device capacitance between these two electrodes, separated by
# the 40 nm mim_diel. TopMetal1 is only the routing metal above the top plate.
sim.add_terminal("T1", layer="metal5")
sim.add_terminal("T2", layer="mim")

sim.set_electrostatic()

print(sim.validate_config())

### Extract the effective permittivity from the PDK

All IHP MIM capacitor models — `cap_cmim`, `cap_rfcmim`, and the design-rule
`mim_cap_density` — report the *same* area-specific capacitance density,
1.5 fF/µm². Since they all describe the same 40 nm `mim_diel` stack, comparing
them is a consistency check, not an independent measurement.

The density implies an *effective* permittivity for the dielectric through the
parallel-plate law `C/A = ε₀·εᵣ/t`:

    εᵣ,eff = (C/A)·t / ε₀

This is a **device-level calibration constant** (the fitted `caspec` may absorb
fringing and parasitic effects), not a bulk material property. To make sure the
extraction step itself is sound, we round-trip it on the **gate oxide**, whose
permittivity the PDK states independently (`sg13g2_moshv_parm.lib`:
`toxo = 7.43 nm`, `epsroxo = 3.9`). Feeding the gate-oxide capacitance density
back through the same formula must recover 3.9 — validating the *arithmetic*
without needing the (unknown) MIM material value.

In [ ]:
# All PDK MIM models report the same 1.5 fF/um^2 area density.
from ihp.tech import CbCapCalc, TECH

EPS0 = 8.854187817e-12  # F/m


def effective_permittivity(cap_density_fF_um2, thickness_um):
    """Back out relative permittivity from a per-area capacitance density."""
    cap = cap_density_fF_um2 * 1e-15 / 1e-12  # fF/um^2 -> F/m^2
    return cap * (thickness_um * 1e-6) / EPS0


print("PDK MIM area-specific capacitance densities (fF/um^2):")
print(f"  cmim_caspec     = {TECH.cmim_caspec}")
print(f"  rfcmim_caspec   = {TECH.rfcmim_caspec}")
print(f"  mim_cap_density = {TECH.mim_cap_density}")
for model in ("cmim", "rfcmim"):
    C = CbCapCalc("C", 0.0, cap_length, cap_width, model)
    print(f"  CbCapCalc({model}, {cap_width}x{cap_length}) = {C:.2f} fF")

# Effective permittivity of the MIM dielectric, from PDK stack thickness.
t_mim = float(PDK.layer_stack.layers["mim_diel"].thickness)  # um
eps_eff = effective_permittivity(TECH.cmim_caspec, t_mim)
print(
    f"\nMIM: caspec={TECH.cmim_caspec} fF/um^2, t={t_mim} um -> eps_eff = {eps_eff:.3f}"
)

# Validate the extraction arithmetic on the gate oxide (independent value).
import re
from pathlib import Path

import ihp

parm_lib = (
    Path(ihp.__file__).parent
    / "models"
    / "ngspice"
    / "models"
    / "sg13g2_moshv_parm.lib"
)
lib_txt = parm_lib.read_text()
_toxo = float(re.search(r"toxo\s*=\s*'([0-9.eE+-]+)", lib_txt).group(1))  # m
_epsroxo = float(re.search(r"epsroxo\s*=\s*([0-9.eE+-]+)", lib_txt).group(1))
cox = _epsroxo * EPS0 / _toxo  # F/m^2
cox_fF = cox * 1e3  # F/m^2 -> fF/um^2 (1 fF/um^2 = 1e-3 F/m^2)
eps_gate = effective_permittivity(cox_fF, _toxo * 1e6)
print(
    f"\ngate oxide: toxo={_toxo * 1e9:.3f} nm, epsroxo={_epsroxo} -> Cox={cox_fF:.3f} fF/um^2"
)
print(f"  round-trip eps = {eps_gate:.3f}  (expect {_epsroxo:.1f})")

### Use the extracted effective permittivity in the simulation

We override the stack's `sin` (SiN) material with the PDK-derived effective
permittivity so the MIM dielectric matches the PDK's fitted area density. This
calibrates the *area* term; what remains — the perimeter/fringe field — is the
quantity the size sweep below probes.

In [ ]:
# Use the PDK-derived effective permittivity for the MIM dielectric (sin).
sim.set_material("sin", material_type="dielectric", permittivity=eps_eff)

### Mesh and generate config

In [ ]:
# Vmim vias are 0.42 um with 0.52 um gaps; MIM dielectric is 40 nm thick.
# Default mesh sizes (refined_mesh_size=5.0 um, max_mesh_size=300 um) keep the
# notebook fast (~seconds/run, <100 MB). For higher accuracy use
# `preset="fine", refined_mesh_size=0.1` but that takes minutes and gigabytes.
sim.mesh(merge_via_distance=0)

# Palace needs a config file to run. `run()` writes it automatically, but the
# local `run_local()` path does not, so write it explicitly after meshing.
sim.write_config()

In [ ]:
# sim.plot_mesh(show_groups=["metal", "topmetal", "via", "dielectric", "SiO2__vmim"])

sim.plot_mesh(show_groups=["metal5", "mim", "topmetal1", "vmim", "SiO2__vmim"])

# sim.plot_mesh(
#     style="solid",
#     transparent_groups=["air__None", "sio2__None", "air__sio2", "air__passive"],
#     interactive=True,
# )

### IHP PDK compact-model reference

This is the value the IHP `cap_cmim` SPICE device would report for the same
geometry. We evaluate the PDK's own area + perimeter law using the live `TECH`
process constants (`cmim_caspec` [fF/um^2], `cmim_cpspec` [fF/um], `cmim_lwd`
[um]) via the PDK's `CbCapCalc` helper, so the reference is authoritative rather
than hand-tuned. This is the number to compare the Palace mutual capacitance
`Cm[1,2]` against.

In [ ]:
# IHP `cap_cmim` compact model: C = caspec*(w+lwd)*(l+lwd) + 2*cpspec*(w+l+2*lwd),
# with the process constants read from the live PDK `TECH` (fF/um^2 and fF/um).
# `CbCapCalc` is the PDK's own area+perimeter evaluator, so the reference is
# authoritative rather than hand-tuned. (The old `ihp.cells2...Numeric` import was
# removed in IHP 2.0.0; `TECH`/`CbCapCalc` is the supported API.)
from ihp.tech import CbCapCalc, TECH

C_pdk = CbCapCalc("C", 0.0, cap_length, cap_width, "cmim")  # fF (total)

# Area + perimeter breakdown (for transparency)
_caspec = TECH.cmim_caspec  # fF/um^2  area-specific capacitance
_cpspec = TECH.cmim_cpspec  # fF/um    perimeter-specific capacitance
_lwd = TECH.cmim_lwd  # um       line-width delta
_leff = cap_length + _lwd
_weff = cap_width + _lwd
C_pdk_area = _caspec * _leff * _weff
C_pdk_perim = 2.0 * (_leff + _weff) * _cpspec

print(f"IHP model 'cap_cmim' for {cap_width} x {cap_length} um cmim:")
print(f"  area term      = {C_pdk_area:7.2f} fF")
print(f"  perimeter term = {C_pdk_perim:7.2f} fF")
print(f"  C_pdk (total)  = {C_pdk:7.2f} fF   <- reference for Palace Cm[1,2]")

### Run

Uncomment the cloud variant to submit to GDSFactory+ cloud. The result should be a
capacitance matrix CSV.

In [ ]:
# Run locally with a Palace executable (works with the installed PDK/fork):
results = sim.run_local()

# Alternatively, submit to the GDSFactory+ cloud:
# results = sim.run()

### Load and analyze results

In [ ]:
import csv
from pathlib import Path

import numpy as np

# `run_local()` returns a dict of file paths, or a PalaceTextResults object
# (with a `.files` mapping) depending on the gsim version.
if isinstance(results, dict):
    terminal_csv = results["terminal-C.csv"]
else:
    terminal_csv = results.files["terminal-C.csv"]
results_dir = Path(terminal_csv).parent


def read_palace_csv(path):
    """Read a Palace output CSV, returning header and data as numpy array."""
    with open(path) as f:
        reader = csv.reader(f)
        header = next(reader)
        data = np.array([[float(x) for x in row] for row in reader])
    return [h.strip() for h in header], data


# Maxwell capacitance matrix (terminal-C.csv): diagonal = self-capacitance,
# off-diagonal = negative induced charge on the grounded neighbor.
header, C_matrix = read_palace_csv(results_dir / "terminal-C.csv")
print("Maxwell capacitance matrix (F):")
print(f"  C[1,1] = {C_matrix[0, 1]:+.4e} F  ({C_matrix[0, 1] * 1e15:+.3f} fF)")
print(f"  C[1,2] = {C_matrix[0, 2]:+.4e} F  ({C_matrix[0, 2] * 1e15:+.3f} fF)")
print(f"  C[2,1] = {C_matrix[1, 1]:+.4e} F  ({C_matrix[1, 1] * 1e15:+.3f} fF)")
print(f"  C[2,2] = {C_matrix[1, 2]:+.4e} F  ({C_matrix[1, 2] * 1e15:+.3f} fF)")

# Mutual (lumped "SPICE") matrix (terminal-Cm.csv): off-diagonal is the physical
# plate-to-plate capacitor. Cm[1,2] is the device capacitance the PDK models.
_, Cm = read_palace_csv(results_dir / "terminal-Cm.csv")
print(f"\nMutual (plate-to-plate) capacitance  Cm[1,2] = {Cm[0, 2] * 1e15:.3f} fF")

# Domain energy
_, E = read_palace_csv(results_dir / "domain-E.csv")
print(
    f"\nStored electric energy: {E[0, 1]:.4e} J (excitation 1), {E[1, 1]:.4e} J (excitation 2)"
)

### Compare: Palace vs IHP PDK model

The Palace plate-to-plate capacitance is `Cm[1,2]` (equivalently `|C[1,2]|`),
compared against the IHP `cap_cmim` compact-model value `C_pdk`.

> **On the residual gap.** We ran Palace with the PDK-derived effective
> permittivity (`eps_eff ≈ 6.8`) in the MIM dielectric, so the *area* term is
> calibrated to `cmim_caspec`. The residual Palace-vs-PDK difference is then
> dominated by the perimeter/fringe field: Palace resolves it in full 3D, while
> the compact model approximates it with a linear `cpspec` term. Because
> fringing is proportionally larger for small plates, the Palace/PDK ratio
> should fall toward 1 as the plate grows — the size sweep below checks this.
>
> A key limitation: Palace's *electrostatic* solver models terminals as ideal
> equipotential (Dirichlet) surfaces and does **not** apply a
> surface-conductivity / conductor-thickness boundary condition. Palace's
> surface-conductivity BC (with a `Thickness` and the standard `t/2`-per-side
> thin-sheet convention) is only available to the *frequency-domain Maxwell*
> solvers, not to capacitance extraction.

In [ ]:
# Palace device (mutual) capacitance — the quantity the PDK models.
C_palace = abs(Cm[0, 2])

print(f"Palace mutual Cm[1,2]:      {C_palace * 1e15:.3f} fF")
print(f"IHP PDK model (cap_cmim):   {C_pdk:.3f} fF")
print(f"Ratio Palace/PDK:           {C_palace * 1e15 / C_pdk:.3f}")

### Size sweep: the remaining gap is fringing

With the effective permittivity in place, the Palace *area* term matches the
PDK's `caspec`. If the residual is a fringing field, the Palace/PDK ratio should
decrease toward 1 as the plate grows (larger area -> smaller perimeter/area
ratio -> smaller relative fringe). We sweep 5, 10, 15, 20, 25 um plates.

The sweep uses a slightly refined mesh (`refined_mesh_size=2.0 um`, still with every via kept separate via `merge_via_distance=0`). The default 5 um mesh degenerates at the 25 um plate (Palace fails with a PETSc NaN), so the finer setting is used for consistency across sizes.


In [ ]:
def run_cmim_cm(width, length, outdir):
    """Mesh + run an ElectrostaticSim and return mutual Cm[1,2] in fF."""
    c = cells.cmim(width=width, length=length).copy()
    s = ElectrostaticSim()
    s.set_output_dir(outdir)
    s.set_geometry(c)
    s.set_stack(substrate_thickness=2.0)
    s.set_airbox(margin_x=5, margin_y=5, z_above=5, z_below=5)
    s.add_terminal("T1", layer="metal5")
    s.add_terminal("T2", layer="mim")
    s.set_electrostatic()
    s.set_material("sin", material_type="dielectric", permittivity=eps_eff)
    s.mesh(refined_mesh_size=2.0, merge_via_distance=0)
    s.write_config()
    res = s.run_local()
    tc = (
        res["terminal-Cm.csv"]
        if isinstance(res, dict)
        else res.files["terminal-Cm.csv"]
    )
    _, cm = read_palace_csv(tc)
    return abs(cm[0, 2]) * 1e15  # fF


sizes = [5.0, 10.0, 15.0, 20.0, 25.0]
sweep = []
for size in sizes:
    c_palace = run_cmim_cm(size, size, f"./palace-sim-electrostatic-{size:g}")
    c_pdk = CbCapCalc("C", 0.0, size, size, "cmim")
    leff = size + TECH.cmim_lwd
    area_fF = TECH.cmim_caspec * leff * leff
    fringe = (c_palace - area_fF) / c_palace * 100
    sweep.append({"size": size, "palace": c_palace, "pdk": c_pdk, "area": area_fF})
    print(
        f"size {size:5.1f} um: Palace={c_palace:8.2f} fF  PDK={c_pdk:8.2f} fF  "
        f"ratio={c_palace / c_pdk:.3f}  fringe={fringe:5.1f}%"
    )

In [ ]:
import matplotlib.pyplot as plt

sizes_np = np.array([r["size"] for r in sweep])
palace = np.array([r["palace"] for r in sweep])
pdk = np.array([r["pdk"] for r in sweep])
area = np.array([r["area"] for r in sweep])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.loglog(sizes_np, palace, "o-", label="Palace (3D)")
ax1.loglog(sizes_np, pdk, "s--", label="IHP cap_cmim")
ax1.loglog(sizes_np, area, ":", color="gray", label="area term caspec*A")
ax1.set_xlabel("plate size (um)")
ax1.set_ylabel("mutual capacitance Cm (fF)")
ax1.set_title("Palace vs PDK vs area term")
ax1.legend()
ax1.grid(True, which="both", alpha=0.3)

ax2.semilogx(sizes_np, palace / pdk, "o-", label="Palace/PDK ratio")
ax2.axhline(1.0, color="gray", ls=":")
ax2.set_xlabel("plate size (um)")
ax2.set_ylabel("Palace / PDK")
ax2.set_title("Residual vs plate size")
ax2.legend()
ax2.grid(True, which="both", alpha=0.3)

fig.tight_layout()
plt.show()

print("\nFringe share of the Palace capacitance (vs ideal area term):")
for r in sweep:
    fringe = (r["palace"] - r["area"]) / r["palace"] * 100
    print(f"  {r['size']:5.1f} um: {fringe:5.1f}%")